# Obligatorio Steam: exploración y limpieza con Spark

## 1. Sesión de Spark y rutas del Data Lake
Abrimos la sesión de Spark y definimos las rutas de las zonas en HDFS.

In [7]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("obligatorio_steam").getOrCreate()

# Zonas del Data Lake
LANDING   = "/lnd/steam"            # datos crudos, tal como los dejó NiFi
REFINADO  = "/rfn/steam/refinado"   # datos limpios en Parquet
MODELO    = "/rfn/steam/modelo"     # tablas del modelo estrella (Hive apunta acá)
ANALITICA = "/anl/steam"            # resultados de las preguntas 4 y 5

print("Spark:", spark.version)

Spark: 3.5.0


## 2. Lectura de los datos de la zona landing
Leemos los cuatro archivos que dejó NiFi en /lnd/steam y verificamos la cantidad de registros de cada uno.

In [8]:
archivos = {
    "games":       "steam_games",
    "users_items": "australian_users_items",
    "reviews":     "australian_user_reviews",
    "bundles":     "bundle_data",
}

crudos = {nombre: spark.read.json(f"{LANDING}/{archivo}.json.gz")
          for nombre, archivo in archivos.items()}

for nombre, df in crudos.items():
    print(f"{nombre:12s} filas: {df.count():>8,}   columnas: {len(df.columns)}")

games        filas:   32,135   columnas: 16
users_items  filas:   88,310   columnas: 5
reviews      filas:   25,799   columnas: 3
bundles      filas:      615   columnas: 7


## 3. Estructura de las tablas
Revisamos las columnas de cada tabla y el tipo de dato que Spark dedujo para cada una.

In [9]:
for nombre, df in crudos.items():
    print(f"========== {nombre} ==========")
    df.printSchema()

========== games ==========
root
 |-- app_name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount_price: double (nullable = true)
 |-- early_access: boolean (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- id: string (nullable = true)
 |-- metascore: string (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- reviews_url: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- specs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)

========== users_items ==========
root
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- item_id: string (nullable = true)
 |    |    |-- it

## 4. Exploración del catálogo de juegos
Miramos algunas filas del catálogo y revisamos los valores de precio que no son números.

In [10]:
games = crudos["games"]

games.select("id", "title", "developer", "price", "release_date",
             "genres", "metascore", "sentiment").show(5, truncate=35)

+------+-----------------------+----------------+------------+------------+-----------------------------------+---------+---------------+
|    id|                  title|       developer|       price|release_date|                             genres|metascore|      sentiment|
+------+-----------------------+----------------+------------+------------+-----------------------------------+---------+---------------+
|761140|    Lost Summoner Kitty|       Kotoshiro|        4.99|  2018-01-04|[Action, Casual, Indie, Simulati...|     NULL|           NULL|
|643980|              Ironbound|Secret Level SRL|Free To Play|  2018-01-04|[Free to Play, Indie, RPG, Strat...|     NULL|Mostly Positive|
|670290|Real Pool 3D - Poolians|    Poolians.com|Free to Play|  2017-07-24|[Casual, Free to Play, Indie, Si...|     NULL|Mostly Positive|
|767400|             弹炸人2222|        彼岸领域|        0.99|  2017-12-07|        [Action, Adventure, Casual]|     NULL|           NULL|
|773570|                   NULL|         

In [11]:
(games
 .filter(F.col("price").isNotNull() & F.col("price").cast("double").isNull())
 .groupBy("price")
 .count()
 .orderBy(F.desc("count"))
 .show(30, truncate=False))

+-----------------------------+-----+
|price                        |count|
+-----------------------------+-----+
|Free                         |905  |
|Free to Play                 |520  |
|Free To Play                 |462  |
|Free Mod                     |4    |
|Free Demo                    |3    |
|Third-party                  |2    |
|Play for Free!               |2    |
|Play Now                     |2    |
|Free Movie                   |1    |
|Play WARMACHINE: Tactics Demo|1    |
|Free HITMAN™ Holiday Pack    |1    |
|Free to Try                  |1    |
|Starting at $449.00          |1    |
|Play the Demo                |1    |
|Starting at $499.00          |1    |
|Install Theme                |1    |
|Install Now                  |1    |
|Free to Use                  |1    |
+-----------------------------+-----+



### Valores nulos y validación de la clave primaria

In [12]:
total = games.count()

fila = games.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                     for c in games.columns]).first()

for c in games.columns:
    print(f"{c:15s} nulos: {fila[c]:>6,}   ({fila[c] / total:.1%})")

app_name        nulos:      2   (0.0%)
developer       nulos:  3,299   (10.3%)
discount_price  nulos: 31,910   (99.3%)
early_access    nulos:      0   (0.0%)
genres          nulos:  3,283   (10.2%)
id              nulos:      2   (0.0%)
metascore       nulos: 29,458   (91.7%)
price           nulos:  1,377   (4.3%)
publisher       nulos:  8,052   (25.1%)
release_date    nulos:  2,067   (6.4%)
reviews_url     nulos:      2   (0.0%)
sentiment       nulos:  7,182   (22.3%)
specs           nulos:    670   (2.1%)
tags            nulos:    163   (0.5%)
title           nulos:  2,050   (6.4%)
url             nulos:      0   (0.0%)


In [13]:
print("Juegos:", total, "| ids distintos:", games.select("id").distinct().count())

games.groupBy("id").count().filter("count > 1").show()

Juegos: 32135 | ids distintos: 32133
+------+-----+
|    id|count|
+------+-----+
|  NULL|    2|
|612880|    2|
+------+-----+



In [14]:
(games
 .filter(F.col("id").isNull() | (F.col("id") == "612880"))
 .select("id", "app_name", "title", "developer", "price", "release_date", "genres", "url")
 .show(truncate=40))

+------+----------------------------------------+----------------------------------------+----------------------------------------+-----+------------+-------------------+----------------------------------------+
|    id|                                app_name|                                   title|                               developer|price|release_date|             genres|                                     url|
+------+----------------------------------------+----------------------------------------+----------------------------------------+-----+------------+-------------------+----------------------------------------+
|  NULL|                                    NULL|                                    NULL|                                    NULL|19.99|        NULL|               NULL|          http://store.steampowered.com/|
|612880|        Wolfenstein II: The New Colossus|        Wolfenstein II: The New Colossus|                           Machine Games|59.99|  2017-10-26|  

In [15]:
games.filter(F.col("id") == "200260").select("id", "app_name").show()

+------+--------------------+
|    id|            app_name|
+------+--------------------+
|200260|Batman: Arkham Ci...|
+------+--------------------+



### Hallazgos y decisiones sobre el catálogo

- **Clave primaria:** hay 32.135 filas y 32.133 `id` distintos.
  - Dos filas tienen `id` nulo. Una no tiene ningún dato y se descarta. La otra es *Batman: Arkham City GOTY*; su URL indica el `id` 200260, que ya existe en otra fila, por lo que es un duplicado y también se descarta.
  - El `id` 612880 (*Wolfenstein II*) aparece dos veces con los mismos datos; se conserva una sola fila.
  - El catálogo limpio queda con 32.132 juegos.
- **Columnas casi vacías:** `discount_price` (99,3 % nulos) y `metascore` (91,7 %) no se usan en el análisis.
- **Nombre del juego:** se usa `app_name`, que está completo, en lugar de `title` (6,4 % nulos).
- **Desarrollador y géneros:** `developer` (10,3 %) y `genres` (10,2 %) tienen nulos; esos juegos quedan fuera de los análisis por desarrollador y por género.
- **Precio:** mezcla números con textos. Regla de limpieza:
  - si es un número, se mantiene;
  - si el texto contiene "free" (sin distinguir mayúsculas), el precio es 0 y el juego se marca como gratuito;
  - si dice "Starting at $X", se toma el valor X;
  - el resto ("Third-party", "Play Now", "Install Now", etc.) queda como precio desconocido.
- **"Free to Play" como género:** aparece dentro de `genres` aunque es un modelo de negocio; se tiene en cuenta en las preguntas por género.

## 5. Exploración de las bibliotecas de usuarios
Revisamos la tabla de usuarios y sus juegos: duplicados, usuarios sin juegos y consistencia de la cantidad declarada.

In [16]:
ui = crudos["users_items"]

ui.select("user_id", "steam_id", "items_count",
          F.size("items").alias("juegos_en_lista")).show(5)

+-----------------+-----------------+-----------+---------------+
|          user_id|         steam_id|items_count|juegos_en_lista|
+-----------------+-----------------+-----------+---------------+
|76561197970982479|76561197970982479|        277|            277|
|          js41637|76561198035864385|        888|            888|
|        evcentric|76561198007712555|        137|            137|
|       Riot-Punch|76561197963445855|        328|            328|
|            doctr|76561198002099482|        541|            541|
+-----------------+-----------------+-----------+---------------+
only showing top 5 rows



In [17]:
filas = ui.count()
print("Filas:", filas, "| user_id distintos:", ui.select("user_id").distinct().count())
print("Filas idénticas repetidas:", filas - ui.dropDuplicates().count())

Filas: 88310 | user_id distintos: 87626
Filas idénticas repetidas: 657


In [18]:
print("Usuarios con la lista de juegos vacía:", ui.filter(F.size("items") == 0).count())
print("items_count distinto del largo de la lista:",
      ui.filter(F.col("items_count") != F.size("items")).count())

Usuarios con la lista de juegos vacía: 16806
items_count distinto del largo de la lista: 0


In [19]:
sin_copias = ui.dropDuplicates()

repetidos = (sin_copias.groupBy("user_id").count()
             .filter("count > 1").select("user_id"))

(sin_copias.join(repetidos, "user_id")
 .select("user_id", "items_count", F.size("items").alias("juegos_en_lista"))
 .orderBy("user_id")
 .show(10))

+-----------------+-----------+---------------+
|          user_id|items_count|juegos_en_lista|
+-----------------+-----------+---------------+
|76561198050680344|        125|            125|
|76561198050680344|        125|            125|
|76561198063371165|         45|             45|
|76561198063371165|         45|             45|
|76561198064484479|        336|            336|
|76561198064484479|        336|            336|
|76561198064956087|         55|             55|
|76561198064956087|         55|             55|
|76561198072861800|          7|              7|
|76561198072861800|          7|              7|
+-----------------+-----------+---------------+
only showing top 10 rows



In [20]:
(sin_copias.join(repetidos, "user_id")
 .select("user_id", "steam_id", "user_url",
         F.aggregate("items", F.lit(0).cast("long"),
                     lambda total, juego: total + juego["playtime_forever"]).alias("minutos_totales"),
         F.aggregate("items", F.lit(0).cast("long"),
                     lambda total, juego: total + juego["playtime_2weeks"]).alias("minutos_2semanas"))
 .orderBy("user_id")
 .show(10, truncate=False))

+-----------------+-----------------+----------------------------------------------------+---------------+----------------+
|user_id          |steam_id         |user_url                                            |minutos_totales|minutos_2semanas|
+-----------------+-----------------+----------------------------------------------------+---------------+----------------+
|76561198050680344|76561198050680344|http://steamcommunity.com/profiles/76561198050680344|240175         |4411            |
|76561198050680344|76561198050680344|http://steamcommunity.com/profiles/76561198050680344|240236         |4472            |
|76561198063371165|76561198063371165|http://steamcommunity.com/profiles/76561198063371165|99563          |2257            |
|76561198063371165|76561198063371165|http://steamcommunity.com/profiles/76561198063371165|99563          |2257            |
|76561198064484479|76561198064484479|http://steamcommunity.com/profiles/76561198064484479|285731         |781             |
|7656119

### Hallazgos y decisiones sobre las bibliotecas de usuarios

- **Usuarios repetidos:** hay 88.310 filas y 87.626 `user_id` distintos (684 filas de más).
  - 657 son copias exactas y se eliminan.
  - En 27 casos el mismo usuario aparece con datos distintos. El `steam_id` y la URL coinciden; solo cambian los minutos jugados, con diferencias muy chicas. Son dos registros de la misma biblioteca tomados en momentos distintos. Se conserva el de más minutos totales y, si empatan, el de más minutos en las últimas dos semanas, ya que las horas jugadas solo aumentan con el tiempo.
- **Usuarios sin juegos:** 16.806 usuarios tienen la biblioteca vacía. Se mantienen en la tabla de usuarios, pero no generan registros de usuario-juego.
- **Consistencia:** `items_count` coincide en todos los casos con la cantidad de juegos de la lista.

## 6. Exploración de las reseñas
Separamos las reseñas de cada usuario en filas individuales y revisamos fechas, duplicados y su relación con el catálogo.

In [21]:
rv = crudos["reviews"]

resenas = (rv.select("user_id", F.explode("reviews").alias("r"))
             .select("user_id", "r.*"))

print("Filas (usuarios):", rv.count(), "| user_id distintos:", rv.select("user_id").distinct().count())
print("Reseñas:", resenas.count())

resenas.select("user_id", "item_id", "posted", "recommend", "helpful", "funny").show(5, truncate=30)

Filas (usuarios): 25799 | user_id distintos: 25485
Reseñas: 59305
+-----------------+-------+-------------------------+---------+------------------------------+-----+
|          user_id|item_id|                   posted|recommend|                       helpful|funny|
+-----------------+-------+-------------------------+---------+------------------------------+-----+
|76561197970982479|   1250| Posted November 5, 2011.|     true|                No ratings yet|     |
|76561197970982479|  22200|    Posted July 15, 2011.|     true|                No ratings yet|     |
|76561197970982479|  43110|   Posted April 21, 2011.|     true|                No ratings yet|     |
|          js41637| 251610|    Posted June 24, 2014.|     true|15 of 20 people (75%) found...|     |
|          js41637| 227300|Posted September 8, 2013.|     true|0 of 1 people (0%) found th...|     |
+-----------------+-------+-------------------------+---------+------------------------------+-----+
only showing top 5 rows



In [22]:
con_anio = F.col("posted").rlike(r"\d{4}")

resenas.groupBy(con_anio.alias("tiene_anio")).count().show()
resenas.filter(~con_anio).select("posted").show(5, truncate=False)

+----------+-----+
|tiene_anio|count|
+----------+-----+
|      true|49186|
|     false|10119|
+----------+-----+

+------------------+
|posted            |
+------------------+
|Posted February 3.|
|Posted May 20.    |
|Posted July 24.   |
|Posted June 16.   |
|Posted June 11.   |
+------------------+
only showing top 5 rows



In [23]:
print("Reseñas repetidas (mismo usuario y juego):",
      resenas.count() - resenas.dropDuplicates(["user_id", "item_id"]).count())

print("Reseñas sin dato de recomendación:", resenas.filter(F.col("recommend").isNull()).count())

ids_catalogo = games.select(F.col("id").alias("item_id")).distinct()
print("Reseñas de juegos que no están en el catálogo:",
      resenas.join(ids_catalogo, "item_id", "left_anti").count())

Reseñas repetidas (mismo usuario y juego): 874
Reseñas sin dato de recomendación: 0
Reseñas de juegos que no están en el catálogo: 5317


In [24]:
print("Reseñas repetidas que son copias exactas:",
      resenas.count() - resenas.dropDuplicates().count())

Reseñas repetidas que son copias exactas: 874


### Hallazgos y decisiones sobre las reseñas

- **Estructura:** cada fila trae la lista de reseñas de un usuario; con `explode` se obtiene una fila por reseña (59.305 en total).
- **Usuarios repetidos:** hay 25.799 filas y 25.485 `user_id` distintos.
- **Reseñas repetidas:** 874 reseñas aparecen dos veces para el mismo usuario y juego. Todas son copias exactas y se eliminan, por lo que quedan 58.431 reseñas.
- **Fechas sin año:** 10.119 reseñas tienen fecha sin año (por ejemplo, "Posted February 3."), porque Steam omite el año en las reseñas del año en curso. Como no se conoce el año de recolección de los datos, esas fechas quedan como desconocidas en lugar de completarse con un año supuesto. La pregunta 3 no depende de la fecha.
- **Recomendación:** la columna `recommend` no tiene nulos.
- **Integridad con el catálogo:** 5.317 reseñas corresponden a juegos que no están en el catálogo. Se conservan, pero no pueden clasificarse por género en la pregunta 3.

## 7. Exploración de los bundles
Revisamos los paquetes de juegos: estructura, formato de los precios y relación de sus juegos con el catálogo.

In [25]:
bundles = crudos["bundles"]

print("Bundles:", bundles.count(), "| bundle_id distintos:", bundles.select("bundle_id").distinct().count())

bundles.select("bundle_id", "bundle_name", "bundle_price", "bundle_final_price",
               "bundle_discount", F.size("items").alias("juegos")).show(5, truncate=30)

Bundles: 615 | bundle_id distintos: 615
+---------+------------------------------+------------+------------------+---------------+------+
|bundle_id|                   bundle_name|bundle_price|bundle_final_price|bundle_discount|juegos|
+---------+------------------------------+------------+------------------+---------------+------+
|      450|  Dharker Studio 2015 Complete|      $73.86|            $66.46|            10%|    14|
|     1473|Naruto Shippuden Uncut Seas...|      $25.87|            $20.15|            22%|    13|
|     1474|Naruto Shippuden Uncut Seas...|      $25.87|            $20.15|            22%|    13|
|     1437|Pixel Puzzles Ultimate: Sta...|      $25.35|            $15.17|            40%|    10|
|     1466|Naruto Shippuden Uncut Seas...|      $25.87|            $20.15|            22%|    13|
+---------+------------------------------+------------+------------------+---------------+------+
only showing top 5 rows



In [26]:
bundle_items = (bundles.select("bundle_id", F.explode("items").alias("it"))
                       .select("bundle_id", "it.*"))

print("Juegos en bundles:", bundle_items.count())
bundle_items.show(5, truncate=30)

Juegos en bundles: 3525
+---------+----------------+------------------------+-------+------------------------------+------------------------------+
|bundle_id|discounted_price|                   genre|item_id|                     item_name|                      item_url|
+---------+----------------+------------------------+-------+------------------------------+------------------------------+
|      450|           $8.99|   Adventure, Indie, RPG| 326950|                Sword of Asumi|http://store.steampowered.c...|
|      450|           $2.99|   Adventure, Indie, RPG| 331490|   Sword of Asumi - Soundtrack|http://store.steampowered.c...|
|      450|           $1.99|   Adventure, Indie, RPG| 331491|Sword of Asumi - Graphic Novel|http://store.steampowered.c...|
|      450|           $0.99|   Adventure, Indie, RPG| 331492|Sword of Asumi - Character ...|http://store.steampowered.c...|
|      450|          $12.99|Adventure, Casual, Indie| 348540|          Divine Slice of Life|http://store.ste

In [27]:
precio = r"^\$\d+(\.\d+)?$"

print("bundle_price con otro formato:")
bundles.filter(~F.col("bundle_price").rlike(precio)).groupBy("bundle_price").count().show(truncate=False)

print("bundle_final_price con otro formato:")
bundles.filter(~F.col("bundle_final_price").rlike(precio)).groupBy("bundle_final_price").count().show(truncate=False)

print("bundle_discount con otro formato:")
bundles.filter(~F.col("bundle_discount").rlike(r"^\d+%$")).groupBy("bundle_discount").count().show(truncate=False)

print("discounted_price de los juegos con otro formato:")
bundle_items.filter(~F.col("discounted_price").rlike(precio)).groupBy("discounted_price").count().show(truncate=False)

bundle_price con otro formato:
+------------+-----+
|bundle_price|count|
+------------+-----+
+------------+-----+

bundle_final_price con otro formato:
+------------------+-----+
|bundle_final_price|count|
+------------------+-----+
+------------------+-----+

bundle_discount con otro formato:
+---------------+-----+
|bundle_discount|count|
+---------------+-----+
|7.5%           |1    |
+---------------+-----+

discounted_price de los juegos con otro formato:
+----------------+-----+
|discounted_price|count|
+----------------+-----+
+----------------+-----+



In [28]:
print("Juegos repetidos dentro de un mismo bundle:",
      bundle_items.count() - bundle_items.dropDuplicates(["bundle_id", "item_id"]).count())

print("Juegos de bundles que no están en el catálogo:",
      bundle_items.join(ids_catalogo, "item_id", "left_anti").count())

Juegos repetidos dentro de un mismo bundle: 4
Juegos de bundles que no están en el catálogo: 512


In [29]:
print("Repetidos que son copias exactas:",
      bundle_items.count() - bundle_items.dropDuplicates().count())

(bundle_items.join(bundle_items.groupBy("bundle_id", "item_id").count().filter("count > 1"),
                   ["bundle_id", "item_id"])
 .select("bundle_id", "item_id", "item_name", "discounted_price")
 .orderBy("bundle_id", "item_id")
 .show(truncate=40))

Repetidos que son copias exactas: 1
+---------+-------+--------------------------------------+----------------+
|bundle_id|item_id|                             item_name|discounted_price|
+---------+-------+--------------------------------------+----------------+
|     1141|       | Battle vs Chess - Floating Island DLC|           $6.99|
|     1141|       |    Battle vs. Chess - Dark Desert DLC|           $6.99|
|       15|       |            GameLoading: OST and eBook|           $6.99|
|       15|       |            GameLoading: Extra Content|           $2.99|
|       19|       |            GameLoading: OST and eBook|           $6.99|
|       19|       |            GameLoading: Extra Content|           $2.99|
|      383|  94620|Hector: Badge of Carnage - Full Series|           $6.66|
|      383|  94620|Hector: Badge of Carnage - Full Series|           $6.66|
+---------+-------+--------------------------------------+----------------+



In [30]:
print("Ítems de bundles sin item_id:",
      bundle_items.filter(F.col("item_id").isNull() | (F.trim(F.col("item_id")) == "")).count())

Ítems de bundles sin item_id: 10


### Hallazgos y decisiones sobre los bundles

- **Clave primaria:** hay 615 bundles y 615 `bundle_id` distintos; no hay repetidos.
- **Estructura:** cada bundle trae la lista de ítems que incluye; con `explode` se obtiene una fila por ítem (3.525 en total).
- **Precios y descuento:** todos los valores de `bundle_price`, `bundle_final_price` y `discounted_price` tienen el formato `$N.NN`, y el descuento el formato `N%` (un caso con decimales: 7.5%). En la limpieza se quitan los símbolos `$` y `%` y se convierten a número decimal.
- **Género:** en los bundles, `genre` es un texto separado por comas, mientras que en el catálogo `genres` es una lista. Para los análisis por género se usan los géneros del catálogo.
- **Ítems sin identificador:** 10 ítems no tienen `item_id`. No pueden relacionarse con el catálogo ni distinguirse entre sí, por lo que se descartan (0,3 % del total).
- **Repetidos:** un ítem (*Hector: Badge of Carnage*, bundle 383) aparece dos veces con los mismos datos; se conserva una sola fila. La relación bundle-juego queda con 3.514 filas.
- **Integridad con el catálogo:** 512 ítems de bundles no están en el catálogo de juegos. Muchos son contenidos adicionales (DLC, bandas sonoras, libros digitales) y no juegos. Se conservan en la relación bundle-juego, pero no pueden cruzarse con los datos del catálogo.

## 8. Refinamiento
Aplicamos las reglas de limpieza definidas en la exploración y guardamos las tablas limpias en formato Parquet en la zona refinada (/rfn/steam/refinado).

### 8.1. Catálogo de juegos

In [31]:
fecha_ok = F.col("release_date").rlike(r"^\d{4}-\d{2}-\d{2}$")

print("Fechas con formato AAAA-MM-DD:", games.filter(fecha_ok).count())

(games.filter(F.col("release_date").isNotNull() & ~fecha_ok)
 .groupBy("release_date").count()
 .orderBy(F.desc("count"))
 .show(15, truncate=False))

Fechas con formato AAAA-MM-DD: 29783
+------------+-----+
|release_date|count|
+------------+-----+
|Coming Soon |27   |
|2018        |21   |
|2017        |20   |
|TBA         |16   |
|Q1 2018     |10   |
|Dec 2017    |8    |
|Coming soon |8    |
|Aug 2014    |7    |
|TBD         |6    |
|Winter 2017 |5    |
|Early 2018  |5    |
|SOON        |5    |
|Fall 2017   |5    |
|Q4 2017     |5    |
|Soon        |4    |
+------------+-----+
only showing top 15 rows



In [32]:
precio_txt   = F.trim(F.col("price"))
precio_num   = precio_txt.cast("double")
precio_desde = F.regexp_extract(precio_txt, r"^Starting at \$(\d+(?:\.\d+)?)$", 1)

juegos = (games
    .filter(F.col("id").isNotNull())
    .dropDuplicates(["id"])
    .select(
        F.col("id").alias("game_id"),
        F.col("app_name").alias("nombre"),
        "developer",
        "publisher",
        F.when(fecha_ok, F.to_date("release_date")).alias("fecha_lanzamiento"),
        F.when(F.col("release_date").rlike(r"\d{4}"),
               F.regexp_extract("release_date", r"(\d{4})", 1).cast("int")).alias("anio_lanzamiento"),
        F.when(precio_num.isNotNull(), precio_num)
         .when(F.lower(precio_txt).contains("free"), F.lit(0.0))
         .when(precio_desde != "", precio_desde.cast("double"))
         .alias("precio"),
        "early_access",
        "sentiment",
        "url"))

juegos = juegos.withColumn("es_gratuito",
                           F.when(F.col("precio").isNotNull(), F.col("precio") == 0))

print("Juegos:", juegos.count())
juegos.groupBy("es_gratuito").count().show()
juegos.printSchema()

Juegos: 32132
+-----------+-----+
|es_gratuito|count|
+-----------+-----+
|       NULL| 1385|
|       true| 1900|
|      false|28847|
+-----------+-----+

root
 |-- game_id: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- fecha_lanzamiento: date (nullable = true)
 |-- anio_lanzamiento: integer (nullable = true)
 |-- precio: double (nullable = true)
 |-- early_access: boolean (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- url: string (nullable = true)
 |-- es_gratuito: boolean (nullable = true)



In [34]:
base = games.filter(F.col("id").isNotNull()).dropDuplicates(["id"])

def lista_a_tabla(columna, nombre_valor):
    return (base
            .select(F.col("id").alias("game_id"), F.explode(columna).alias(nombre_valor))
            .withColumn(nombre_valor, F.trim(F.regexp_replace(F.col(nombre_valor), "&amp;", "&")))
            .filter(F.col(nombre_valor) != "")
            .dropDuplicates())

juego_genero = lista_a_tabla("genres", "genero")
juego_tag    = lista_a_tabla("tags", "tag")
juego_spec   = lista_a_tabla("specs", "spec")

for nombre, df in [("juego_genero", juego_genero), ("juego_tag", juego_tag), ("juego_spec", juego_spec)]:
    print(f"{nombre:13s} filas: {df.count():>7,}")

juego_genero.groupBy("genero").count().orderBy(F.desc("count")).show(25, truncate=False)

print("Juegos con &amp; en nombre, developer o publisher:",
      juegos.filter(F.col("nombre").contains("&amp;") |
                    F.col("developer").contains("&amp;") |
                    F.col("publisher").contains("&amp;")).count())

juego_genero  filas:  71,551
juego_tag     filas: 169,189
juego_spec    filas: 143,414
+---------------------+-----+
|genero               |count|
+---------------------+-----+
|Indie                |15858|
|Action               |11319|
|Casual               |8282 |
|Adventure            |8242 |
|Strategy             |6957 |
|Simulation           |6699 |
|RPG                  |5479 |
|Free to Play         |2031 |
|Early Access         |1462 |
|Sports               |1257 |
|Massively Multiplayer|1108 |
|Racing               |1083 |
|Design & Illustration|460  |
|Utilities            |340  |
|Web Publishing       |268  |
|Animation & Modeling |183  |
|Education            |125  |
|Video Production     |116  |
|Software Training    |105  |
|Audio Production     |93   |
|Photo Editing        |77   |
|Accounting           |7    |
+---------------------+-----+

Juegos con &amp; en nombre, developer o publisher: 141


In [35]:
for c in ["nombre", "developer", "publisher"]:
    juegos = juegos.withColumn(c, F.regexp_replace(F.col(c), "&amp;", "&"))

entidad_html = r"&(#\d+|[a-zA-Z]+);"

print("Juegos que todavía tienen alguna entidad HTML:",
      juegos.filter(F.col("nombre").rlike(entidad_html) |
                    F.col("developer").rlike(entidad_html) |
                    F.col("publisher").rlike(entidad_html)).count())

(juegos.filter(F.col("nombre").rlike(entidad_html))
 .select("game_id", "nombre").show(10, truncate=False))

Juegos que todavía tienen alguna entidad HTML: 37
+-------+-----------------------------------------+
|game_id|nombre                                   |
+-------+-----------------------------------------+
|8300   |Sam & Max 205: What&#146;s New Beelzebub?|
+-------+-----------------------------------------+



In [36]:
import html

desescapar = F.udf(lambda texto: html.unescape(texto) if texto is not None else None, "string")

for c in ["nombre", "developer", "publisher"]:
    juegos = juegos.withColumn(c, desescapar(F.col(c)))

entidad_html = r"&(#\d+|[a-zA-Z]+);"

print("Juegos que todavía tienen alguna entidad HTML:",
      juegos.filter(F.col("nombre").rlike(entidad_html) |
                    F.col("developer").rlike(entidad_html) |
                    F.col("publisher").rlike(entidad_html)).count())

juegos.filter(F.col("game_id") == "8300").select("game_id", "nombre").show(truncate=False)

Juegos que todavía tienen alguna entidad HTML: 0
+-------+------------------------------------+
|game_id|nombre                              |
+-------+------------------------------------+
|8300   |Sam & Max 205: What’s New Beelzebub?|
+-------+------------------------------------+



In [37]:
tablas_catalogo = {
    "juegos":       juegos,
    "juego_genero": juego_genero,
    "juego_tag":    juego_tag,
    "juego_spec":   juego_spec,
}

for nombre, df in tablas_catalogo.items():
    ruta = f"{REFINADO}/{nombre}"
    df.write.mode("overwrite").parquet(ruta)
    print(f"{nombre:13s} guardada en {ruta}: {spark.read.parquet(ruta).count():>7,} filas")

juegos        guardada en /rfn/steam/refinado/juegos:  32,132 filas
juego_genero  guardada en /rfn/steam/refinado/juego_genero:  71,551 filas
juego_tag     guardada en /rfn/steam/refinado/juego_tag: 169,189 filas
juego_spec    guardada en /rfn/steam/refinado/juego_spec: 143,414 filas


### 8.2. Bibliotecas de usuarios

In [38]:
from pyspark.sql import Window

sumar = lambda campo: F.aggregate("items", F.lit(0).cast("long"),
                                  lambda total, juego: total + juego[campo])

ui_unicos = (ui.dropDuplicates()
               .withColumn("min_totales", sumar("playtime_forever"))
               .withColumn("min_2semanas", sumar("playtime_2weeks")))

ventana = Window.partitionBy("user_id").orderBy(F.desc("min_totales"), F.desc("min_2semanas"))

ui_limpio = (ui_unicos
             .withColumn("orden", F.row_number().over(ventana))
             .filter(F.col("orden") == 1)
             .drop("orden", "min_totales", "min_2semanas"))

usuarios = ui_limpio.select("user_id", "steam_id", "user_url",
                            F.col("items_count").alias("cantidad_juegos"))

usuario_juego = (ui_limpio
                 .select("user_id", F.explode("items").alias("it"))
                 .select("user_id",
                         F.col("it.item_id").alias("game_id"),
                         F.col("it.playtime_forever").alias("minutos_totales"),
                         F.col("it.playtime_2weeks").alias("minutos_2semanas")))

print("Usuarios:", usuarios.count(), "| user_id distintos:", usuarios.select("user_id").distinct().count())
print("Filas usuario-juego:", usuario_juego.count())
print("Pares usuario-juego repetidos:",
      usuario_juego.count() - usuario_juego.dropDuplicates(["user_id", "game_id"]).count())
print("Filas de juegos que no están en el catálogo:",
      usuario_juego.join(juegos.select("game_id"), "game_id", "left_anti").count())
usuario_juego.show(5)

Usuarios: 87626 | user_id distintos: 87626
Filas usuario-juego: 5094082
Pares usuario-juego repetidos: 0
Filas de juegos que no están en el catálogo: 849270
+----------------+-------+---------------+----------------+
|         user_id|game_id|minutos_totales|minutos_2semanas|
+----------------+-------+---------------+----------------+
|-AnimeIsMyThing-|    240|           6119|               0|
|-AnimeIsMyThing-|   4000|          63967|             206|
|-AnimeIsMyThing-|  10090|           1373|               0|
|-AnimeIsMyThing-|  17390|            504|               0|
|-AnimeIsMyThing-|    550|             47|               0|
+----------------+-------+---------------+----------------+
only showing top 5 rows



In [39]:
faltantes = usuario_juego.join(juegos.select("game_id"), "game_id", "left_anti")

nombres = (ui.select(F.explode("items").alias("it"))
             .select(F.col("it.item_id").alias("game_id"), F.col("it.item_name").alias("nombre"))
             .dropDuplicates(["game_id"]))

print("Juegos distintos que no están en el catálogo:", faltantes.select("game_id").distinct().count())

total_min = usuario_juego.agg(F.sum("minutos_totales")).first()[0]
falt_min  = faltantes.agg(F.sum("minutos_totales")).first()[0]
print(f"Minutos jugados de esos juegos: {falt_min:,} de {total_min:,} ({falt_min / total_min:.1%})")

(faltantes.groupBy("game_id")
 .agg(F.count("*").alias("usuarios"), F.sum("minutos_totales").alias("minutos"))
 .join(nombres, "game_id", "left")
 .orderBy(F.desc("minutos"))
 .show(15, truncate=40))

Juegos distintos que no están en el catálogo: 1786
Minutos jugados de esos juegos: 583,513,105 de 5,045,724,159 (11.6%)
+-------+--------+---------+----------------------------------------+
|game_id|usuarios|  minutos|                                  nombre|
+-------+--------+---------+----------------------------------------+
|  72850|   22038|134851450|             The Elder Scrolls V: Skyrim|
| 202990|    7437| 38209064|Call of Duty: Black Ops II - Multiplayer|
|  10190|    5907| 37930727|Call of Duty: Modern Warfare 2 - Mult...|
|  42690|    4944| 25161489|Call of Duty: Modern Warfare 3 - Mult...|
|  42710|    5568| 24303252|   Call of Duty: Black Ops - Multiplayer|
| 238960|   12820| 23061040|                           Path of Exile|
|  34330|    5534| 14989608|                     Total War: SHOGUN 2|
| 244850|    7690| 13750903|                         Space Engineers|
| 243870|   14449| 13459999|  Tom Clancy's Ghost Recon Phantoms - NA|
|  12210|    9924| 12877829|            

In [40]:
tablas_usuarios = {
    "usuarios":      usuarios,
    "usuario_juego": usuario_juego,
}

for nombre, df in tablas_usuarios.items():
    ruta = f"{REFINADO}/{nombre}"
    df.write.mode("overwrite").parquet(ruta)
    print(f"{nombre:14s} guardada en {ruta}: {spark.read.parquet(ruta).count():>9,} filas")

usuarios       guardada en /rfn/steam/refinado/usuarios:    87,626 filas
usuario_juego  guardada en /rfn/steam/refinado/usuario_juego: 5,094,082 filas


### 8.3. Reseñas

In [41]:
resenas_limpias = (resenas
    .dropDuplicates()
    .withColumn("fecha_txt", F.regexp_extract("posted", r"Posted (\w+ \d{1,2}, \d{4})", 1))
    .select("user_id",
            F.col("item_id").alias("game_id"),
            F.when(F.col("fecha_txt") != "", F.to_date("fecha_txt", "MMMM d, yyyy")).alias("fecha_publicacion"),
            F.col("recommend").alias("recomienda"),
            F.col("review").alias("texto")))

print("Reseñas:", resenas_limpias.count())
print("Con fecha:", resenas_limpias.filter(F.col("fecha_publicacion").isNotNull()).count(),
      "| sin fecha:", resenas_limpias.filter(F.col("fecha_publicacion").isNull()).count())
resenas_limpias.agg(F.min("fecha_publicacion"), F.max("fecha_publicacion")).show()
resenas_limpias.groupBy("recomienda").count().show()
print("Reseñas de usuarios que no están en la tabla usuarios:",
      resenas_limpias.join(usuarios.select("user_id"), "user_id", "left_anti").count())
resenas_limpias.select("user_id", "game_id", "fecha_publicacion", "recomienda").show(5)

Reseñas: 58431
Con fecha: 48498 | sin fecha: 9933
+----------------------+----------------------+
|min(fecha_publicacion)|max(fecha_publicacion)|
+----------------------+----------------------+
|            2010-10-16|            2015-12-31|
+----------------------+----------------------+

+----------+-----+
|recomienda|count|
+----------+-----+
|      true|51692|
|     false| 6739|
+----------+-----+

Reseñas de usuarios que no están en la tabla usuarios: 15
+-----------------+-------+-----------------+----------+
|          user_id|game_id|fecha_publicacion|recomienda|
+-----------------+-------+-----------------+----------+
|76561198054465623| 105600|       2015-07-13|      true|
|76561198164545985|    730|             NULL|      true|
|           BBBirb|   4000|       2014-01-07|      true|
|         BugsyBro|   4000|       2014-06-29|      true|
|           C14M8S|    570|       2014-05-09|     false|
+-----------------+-------+-----------------+----------+
only showing top 5 rows

In [42]:
ruta = f"{REFINADO}/resenas"
resenas_limpias.write.mode("overwrite").parquet(ruta)
print(f"resenas guardada en {ruta}: {spark.read.parquet(ruta).count():,} filas")

resenas guardada en /rfn/steam/refinado/resenas: 58,431 filas


**Resultado del refinamiento de las reseñas:**
- `resenas`: 58.431 reseñas (se eliminaron 874 copias exactas), con `user_id`, `game_id`, `fecha_publicacion`, `recomienda` y `texto`.
- **Fechas:** 48.498 reseñas tienen fecha completa, entre el 16/10/2010 y el 31/12/2015. Las 9.933 restantes no tienen año y quedan sin fecha. Como la última fecha con año es el 31/12/2015 y Steam omite el año en las reseñas del año en curso, es probable que los datos se hayan recolectado en 2016 y que esas reseñas sean de ese año; no se completa el año porque es una deducción y no un dato.
- **Recomendación:** 51.692 reseñas recomiendan el juego (88,5 %) y 6.739 no.
- **Integridad:** 15 reseñas son de usuarios que no están en la tabla `usuarios`; se conservan.

### 8.4. Bundles

In [43]:
a_numero = lambda columna: F.regexp_replace(F.col(columna), r"[$%,]", "").cast("double")

bundles_limpios = bundles.select(
    "bundle_id",
    desescapar(F.col("bundle_name")).alias("nombre"),
    a_numero("bundle_price").alias("precio_original"),
    a_numero("bundle_final_price").alias("precio_final"),
    a_numero("bundle_discount").alias("descuento_pct"),
    F.col("bundle_url").alias("url"))

bundle_juego = (bundle_items
    .filter(F.col("item_id").isNotNull() & (F.trim(F.col("item_id")) != ""))
    .dropDuplicates(["bundle_id", "item_id"])
    .select("bundle_id",
            F.col("item_id").alias("game_id"),
            desescapar(F.col("item_name")).alias("nombre_item"),
            a_numero("discounted_price").alias("precio_con_descuento")))

for nombre, df in [("bundles", bundles_limpios), ("bundle_juego", bundle_juego)]:
    ruta = f"{REFINADO}/{nombre}"
    df.write.mode("overwrite").parquet(ruta)
    print(f"{nombre:13s} guardada en {ruta}: {spark.read.parquet(ruta).count():,} filas")

print("Precios o descuentos que no se pudieron convertir:",
      bundles_limpios.filter(F.col("precio_original").isNull() | F.col("precio_final").isNull()
                             | F.col("descuento_pct").isNull()).count())
bundles_limpios.filter(F.col("descuento_pct") == 7.5).show(truncate=30)

bundles       guardada en /rfn/steam/refinado/bundles: 615 filas
bundle_juego  guardada en /rfn/steam/refinado/bundle_juego: 3,514 filas
Precios o descuentos que no se pudieron convertir: 0
+---------+-------------------+---------------+------------+-------------+------------------------------+
|bundle_id|             nombre|precio_original|precio_final|descuento_pct|                           url|
+---------+-------------------+---------------+------------+-------------+------------------------------+
|      903|Valley + Soundtrack|          26.98|       24.96|          7.5|http://store.steampowered.c...|
+---------+-------------------+---------------+------------+-------------+------------------------------+



### Resumen del refinamiento

Las tablas limpias quedaron guardadas en formato Parquet en `/rfn/steam/refinado`:

| Tabla | Filas | Contenido |
|---|---|---|
| `juegos` | 32.132 | Catálogo: nombre, desarrollador, editor, fecha y año de lanzamiento, precio, si es gratuito, early access, sentimiento y URL |
| `juego_genero` | 71.551 | Un género por fila para cada juego |
| `juego_tag` | 169.189 | Una etiqueta de usuarios por fila para cada juego |
| `juego_spec` | 143.414 | Una característica técnica por fila para cada juego |
| `usuarios` | 87.626 | Usuarios únicos con su cantidad de juegos |
| `usuario_juego` | 5.094.082 | Un juego de la biblioteca de un usuario por fila, con los minutos jugados en total y en las últimas dos semanas |
| `resenas` | 58.431 | Reseñas con fecha de publicación, recomendación y texto |
| `bundles` | 615 | Paquetes con precio original, precio final y porcentaje de descuento |
| `bundle_juego` | 3.514 | Un ítem de un bundle por fila, con su precio con descuento |

Las listas anidadas de los archivos originales se separaron en tablas propias, los números y las fechas quedaron con su tipo correcto, se eliminaron duplicados y registros sin identificador, y se corrigieron las entidades HTML en los textos.

## 9. Diccionario de datos

Columnas y tipos de las tablas guardadas en la zona refinada.

In [44]:
tablas = ["juegos", "juego_genero", "juego_tag", "juego_spec", "usuarios",
          "usuario_juego", "resenas", "bundles", "bundle_juego"]

for nombre in tablas:
    print(f"== {nombre} ==")
    for campo in spark.read.parquet(f"{REFINADO}/{nombre}").schema.fields:
        print(f"   {campo.name:22s} {campo.dataType.simpleString()}")
    print()

== juegos ==
   game_id                string
   nombre                 string
   developer              string
   publisher              string
   fecha_lanzamiento      date
   anio_lanzamiento       int
   precio                 double
   early_access           boolean
   sentiment              string
   url                    string
   es_gratuito            boolean

== juego_genero ==
   game_id                string
   genero                 string

== juego_tag ==
   game_id                string
   tag                    string

== juego_spec ==
   game_id                string
   spec                   string

== usuarios ==
   user_id                string
   steam_id               string
   user_url               string
   cantidad_juegos        bigint

== usuario_juego ==
   user_id                string
   game_id                string
   minutos_totales        bigint
   minutos_2semanas       bigint

== resenas ==
   user_id                string
   game_id               

### Diccionario de datos de la zona refinada

**`juegos`**: una fila por juego del catálogo de Steam.

| Columna | Tipo | Significado |
|---|---|---|
| `game_id` | string | Identificador del juego en Steam (app id). Clave primaria |
| `nombre` | string | Nombre del juego |
| `developer` | string | Empresa o persona que desarrolló el juego. Puede traer varias separadas por coma |
| `publisher` | string | Empresa que publicó el juego |
| `fecha_lanzamiento` | date | Fecha de lanzamiento. Vacía si el original no tenía una fecha completa |
| `anio_lanzamiento` | int | Año de lanzamiento, extraído aunque la fecha estuviera incompleta |
| `precio` | double | Precio en dólares. 0 si es gratuito y vacío si no se pudo determinar |
| `early_access` | boolean | Indica si el juego estaba en acceso anticipado |
| `sentiment` | string | Valoración general de las reseñas en Steam (por ejemplo, "Very Positive") |
| `url` | string | Página del juego en la tienda de Steam |
| `es_gratuito` | boolean | Verdadero si el precio es 0. Vacío si el precio es desconocido |

**`juego_genero`**, **`juego_tag`** y **`juego_spec`**: una fila por cada par juego-valor. Clave primaria: las dos columnas.

| Columna | Tipo | Significado |
|---|---|---|
| `game_id` | string | Juego al que pertenece el valor |
| `genero` | string | Género asignado por Steam (por ejemplo, Action o Indie) |
| `tag` | string | Etiqueta que le pusieron los usuarios (por ejemplo, Open World) |
| `spec` | string | Característica técnica (por ejemplo, Single-player o Steam Achievements) |

**`usuarios`**: una fila por usuario.

| Columna | Tipo | Significado |
|---|---|---|
| `user_id` | string | Identificador del usuario: su ID numérico o el alias de su perfil. Clave primaria |
| `steam_id` | string | ID numérico de 17 dígitos de la cuenta de Steam |
| `user_url` | string | Dirección del perfil público del usuario |
| `cantidad_juegos` | bigint | Cantidad de juegos que tiene en su biblioteca |

**`usuario_juego`**: una fila por juego en la biblioteca de un usuario. Clave primaria: `user_id` + `game_id`.

| Columna | Tipo | Significado |
|---|---|---|
| `user_id` | string | Usuario dueño del juego |
| `game_id` | string | Juego de la biblioteca |
| `minutos_totales` | bigint | Minutos jugados en total |
| `minutos_2semanas` | bigint | Minutos jugados en las dos semanas previas a la recolección de los datos |

**`resenas`**: una fila por reseña.

| Columna | Tipo | Significado |
|---|---|---|
| `user_id` | string | Usuario que escribió la reseña |
| `game_id` | string | Juego reseñado |
| `fecha_publicacion` | date | Fecha de publicación. Vacía si el original no indicaba el año |
| `recomienda` | boolean | Verdadero si el usuario recomienda el juego |
| `texto` | string | Texto de la reseña |

**`bundles`**: una fila por paquete.

| Columna | Tipo | Significado |
|---|---|---|
| `bundle_id` | string | Identificador del bundle en Steam. Clave primaria |
| `nombre` | string | Nombre del bundle |
| `precio_original` | double | Suma de los precios de sus ítems, en dólares |
| `precio_final` | double | Precio del bundle con el descuento aplicado, en dólares |
| `descuento_pct` | double | Porcentaje de descuento del bundle |
| `url` | string | Página del bundle en la tienda |

**`bundle_juego`**: una fila por ítem incluido en un bundle. Clave primaria: `bundle_id` + `game_id`.

| Columna | Tipo | Significado |
|---|---|---|
| `bundle_id` | string | Bundle que contiene el ítem |
| `game_id` | string | Ítem incluido. Puede ser un juego, un DLC o una banda sonora |
| `nombre_item` | string | Nombre del ítem |
| `precio_con_descuento` | double | Precio del ítem dentro del bundle, en dólares |